In [2]:
import pandas as pd
import plotly.express as px

In [31]:
df = pd.read_csv("Compute allocations.csv")
# byte_level = df[df['Patch size'] == 1]
byte_level = df[(df['FLOPs'] == 's') & (df['Patch size'] == 1) & (df['Global transformer layers'] >= 6)]
# byte_level['Global transformer parameters'].sort_values(ascending=False)
fig = px.histogram(byte_level['Global transformer parameters'])

fig.show()

In [32]:
byte_level[(byte_level['Global transformer parameters'] <= 3e7)].head(20)

,FLOPs,Patch size,Tokens,Encoder parameters,Global transformer parameters,Decoder parameters,Global transformer dimension,Encoder/Decoder dimension,Global transformer layers,Decoder layers
74,s,1,1.094277e+10,198272,25219072,991360,512,128,8,5
275,s,1,1.087832e+10,198272,25219072,1189632,512,128,8,6
476,s,1,1.081462e+10,198272,25219072,1387904,512,128,8,7
673,s,1,1.075167e+10,198272,25219072,1586176,512,128,8,8
870,s,1,1.068944e+10,198272,25219072,1784448,512,128,8,9


In [26]:
from scaling_sweep import forward_blt_flops
forward_flops1 = forward_blt_flops(8192, patch_size=4, hidden_state_g=4*128, layers_g=8, hidden_state_e=2*64, layers_e=1, window_e=512, hidden_state_d=2*64, layers_d=5, window_d=512, ratio_patchdim2bytedim=2, n_heads_e=2, n_heads_g=4, n_heads_d=2, vocab=256, feed_forward_mult=4)
forward_flops2 = forward_blt_flops(8192, patch_size=1, hidden_state_g=4*128, layers_g=8, hidden_state_e=2*64, layers_e=1, window_e=512, hidden_state_d=2*64, layers_d=5, window_d=512, ratio_patchdim2bytedim=2, n_heads_e=2, n_heads_g=4, n_heads_d=2, vocab=256, feed_forward_mult=4)
forward_flops2/forward_flops1

# 1.09e10 tokens on patch size 1
# 2.9e10 tokens on patch size 2


5.918771878555751

Good, this is the wide distribution of tokens:params we wanted to emulate (Evo varied between 50M to 120M on 8e18 FLOPs)

Since we're scaling with the additional dimension of patch_size, let's select 3 runs per patch size and only assess 3 varying scales. If the sharper tokens:params distributions perform better, it's a sign that we should use a more aggressive patching scheme.

4e18 FLOPs
===

Patch size = 1

522: 4e9 tokens, 85M parameters, 12 global transformer layers

452: 8e9 tokens, 42M parameters, 6 global transformer layers

74: 1e10 tokens, 25M parameters, 8 global transformer layers


Patch size = 2

1791: 4.9e9 tokens, 200M parameters, 16 global transformer layers

1111: 9.6e9 tokens, 100M parameters, 8 global transformer layers

1103: 3e10 tokens, 25M parameters, 8 global transformer layers


Patch size = 4

6221: 4e9 tokens, 500M parameters, 18 global transformer layers

3954: 8.8e9 tokens, 250M parameters, 20 global transformer layers

3496: 6.5e10 tokens, 25M parameters, 8 global transformer layers




wait whats going on rn why cant we just use the 470m param setting they used with patch size 4? was somethng wrong with my scaling sweep?

470M global latent
1280 global hidden dimension